# ALPSS Results Analysis

Compile shock experiment results from CSV files into a pandas DataFrame for analysis. This notebook queries all ALPSS results, applies type coercion to numeric columns, and shows how to explore the data.

In [1]:
import sys
from pathlib import Path

import pandas as pd
import requests

sys.path.insert(0, str(Path.cwd().parent / 'src'))
from aimdl_examples import (
    get_girder_client,
    fetch_and_parse,
    coerce_types,
    paginate_datafiles,
    enrich_alpss_with_material_properties,
)

## Fetch Results

Query all `pdv_alpss_result` items from the API, parse each CSV file, and combine into a single DataFrame.

In [2]:
with requests.Session() as session:
    gc = get_girder_client(session=session)
    rows = paginate_datafiles(
        gc,
        "pdv_alpss_result",
        lambda item: fetch_and_parse(gc, item),
        extraFields='["meta.prov", "meta.runId"]'
    )

    df = pd.DataFrame(rows)
    df = coerce_types(df)

In [3]:
# Add material properties from the entry form
df = enrich_alpss_with_material_properties(df, gc)

print(f"Added material property columns:")
print(f"  material_c0: {df['material_c0'].notna().sum()} / {len(df)} rows")
print(f"  material_c_l: {df['material_c_l'].notna().sum()} / {len(df)} rows")
print(f"  material_density: {df['material_density'].notna().sum()} / {len(df)} rows")

Added material property columns:
  material_c0: 1912 / 3297 rows
  material_c_l: 1912 / 3297 rows
  material_density: 1912 / 3297 rows


## Enrich with material properties

Query the material properties form and add columns for c0, c_l, and density.

### Data Overview

Each row is one shock experiment result. Example of columns:
- **`igsn`** — Sample identifier
- **`Velocity OK`** — Whether the velocity computation step passed quality checks (True/False)
- **`Velocity at Max Compression`** — Peak velocity (m/s)
- **`Spall OK`, `Spall Strength`, `Strain Rate`** - Spall quality check and derived metrics
- **`HEL OK`,`HEL Strength (GPa)`, `HEL Free Surface Velocity`** - HEL quality check and derived metrics
- Uncertainty columns (e.g., `Velocity at Max Compression Vel Uncertainty`, `Spall Strength Uncertainty`) quantify measurement error
- **`alpss_version`, `alpss_dagster_version`** — Processing pipeline versions

In [4]:
print(f"Total results: {len(df)}")
print(f"Columns: {len(df.columns)}")
print(f"\nShape: {df.shape}")

Total results: 3297
Columns: 46

Shape: (3297, 46)


## Explore Results

View sample data:

In [5]:
df.head()

,Date,File Name,Run Time,Error Message,Velocity OK,Velocity at Max Compression,Time at Max Compression,Velocity at Max Compression Freq Uncertainty,Velocity at Max Compression Vel Uncertainty,Carrier Frequency,...,HEL Strain Rate,Peak Shock Stress,igsn,itemId,runId,alpss_dagster_version,alpss_version,material_c0,material_c_l,material_density
0,Jun 11 2026 04:42 PM,C1--20250807--00002.csv,0 days 00:00:00.800565,velocity: Velocity 2.1699861103704454 did not ...,False,2.169986,0.000002,8.506122e+05,0.659224,1.852000e+09,...,NaN,NaN,JHAMAA00001-S5R4C3,6a296cbaac4f2d98ffa04677,ac25c1d9-ebdf-4bdc-9ce1-b692c2e78a7e,0.1.1,1.7.1,<NA>,<NA>,<NA>
1,Jun 11 2026 04:43 PM,C1--20250807--00009.csv,0 days 00:00:00.783401,velocity: Velocity 1.013024448780469 did not e...,False,1.013024,0.000002,1.813703e+06,1.405619,1.860058e+09,...,NaN,NaN,JHAMAA00001-S5R4C3,6a296ced0b116ccbfdcc2125,ac25c1d9-ebdf-4bdc-9ce1-b692c2e78a7e,0.1.1,1.7.1,<NA>,<NA>,<NA>
2,Jun 11 2026 04:42 PM,C1--20250807--00005.csv,0 days 00:00:00.794346,velocity: Velocity 2.317337597437216 did not e...,False,2.317338,0.000002,1.345923e+06,1.043090,1.824000e+09,...,NaN,NaN,JHAMAA00001-S5R4C3,6a296cd00b116ccbfdcc200b,ac25c1d9-ebdf-4bdc-9ce1-b692c2e78a7e,0.1.1,1.7.1,<NA>,<NA>,<NA>
3,Jun 11 2026 04:43 PM,C1--20250807--00008.csv,0 days 00:00:00.798942,velocity: Velocity 2.379953845231893 did not e...,False,2.379954,0.000002,1.343252e+06,1.041020,1.840000e+09,...,NaN,NaN,JHAMAA00001-S5R4C3,6a296ce60b116ccbfdcc2084,ac25c1d9-ebdf-4bdc-9ce1-b692c2e78a7e,0.1.1,1.7.1,<NA>,<NA>,<NA>
4,Jun 11 2026 04:42 PM,C1--20250807--00004.csv,0 days 00:00:00.811555,velocity: Velocity 3.117297465239656 did not e...,False,3.117297,0.000002,1.323286e+06,1.025547,1.796000e+09,...,NaN,NaN,JHAMAA00001-S5R4C3,6a296cc90b116ccbfdcc1f72,ac25c1d9-ebdf-4bdc-9ce1-b692c2e78a7e,0.1.1,1.7.1,<NA>,<NA>,<NA>


### Filter by Quality

`Velocity OK` indicates whether ALPSS validation of the velocity computation step passed. Similar flags exist for HEL and Spall. View only passing results:

In [6]:
df_passing = df[df['Velocity OK'] == True]
print(f"Passing results: {len(df_passing)} / {len(df)} ({100*len(df_passing)/len(df):.1f}%)")
df_passing.head()

Passing results: 1100 / 3297 (33.4%)


,Date,File Name,Run Time,Error Message,Velocity OK,Velocity at Max Compression,Time at Max Compression,Velocity at Max Compression Freq Uncertainty,Velocity at Max Compression Vel Uncertainty,Carrier Frequency,...,HEL Strain Rate,Peak Shock Stress,igsn,itemId,runId,alpss_dagster_version,alpss_version,material_c0,material_c_l,material_density
25,Jun 11 2026 04:41 PM,C1--20250807--00028.csv,0 days 00:00:00.899024,spall: skipping velocity_ok=True bool(props)=F...,True,342.820936,0.000002,7.367634e+05,0.570992,1.840058e+09,...,NaN,NaN,JHAMAA00001-S5R3C3,6a296c6c40fe08cc73d1001c,2e420028-11b2-4d5d-8ee2-91826e25c5c1,0.1.1,1.7.1,<NA>,<NA>,<NA>
29,Jun 11 2026 04:40 PM,C1--20250807--00026.csv,0 days 00:00:00.934029,spall: skipping velocity_ok=True bool(props)=F...,True,353.049451,0.000002,6.699594e+05,0.519219,1.776056e+09,...,NaN,NaN,JHAMAA00001-S5R3C3,6a296c5d40fe08cc73d0ff8a,2e420028-11b2-4d5d-8ee2-91826e25c5c1,0.1.1,1.7.1,<NA>,<NA>,<NA>
35,Jun 11 2026 04:41 PM,C1--20250807--00034.csv,0 days 00:00:00.884988,spall: skipping velocity_ok=True bool(props)=F...,True,332.378504,0.000002,1.487001e+06,1.152426,1.812057e+09,...,NaN,NaN,JHAMAA00001-S5R3C3,6a296c980b116ccbfdcc1e4f,2e420028-11b2-4d5d-8ee2-91826e25c5c1,0.1.1,1.7.1,<NA>,<NA>,<NA>
36,Jun 11 2026 04:41 PM,C1--20250807--00035.csv,0 days 00:00:00.902515,spall: skipping velocity_ok=True bool(props)=F...,True,350.623525,0.000002,1.321728e+06,1.024339,1.836000e+09,...,NaN,NaN,JHAMAA00001-S5R3C3,6a296c9f20783bdc81a18307,2e420028-11b2-4d5d-8ee2-91826e25c5c1,0.1.1,1.7.1,<NA>,<NA>,<NA>
42,Jun 11 2026 04:42 PM,C1--20250807--00044.csv,0 days 00:00:00.769468,spall: skipping velocity_ok=True bool(props)=F...,True,60.828302,0.000002,2.630700e+06,2.038793,1.852000e+09,...,NaN,NaN,JHAMAA00001-S5R3C3,6a296ce020783bdc81a1842d,2e420028-11b2-4d5d-8ee2-91826e25c5c1,0.1.1,1.7.1,<NA>,<NA>,<NA>


### Filter by Material Properties

Material properties, such as density, longitudinal wave speed, or bulk wave speed, are pulled from the portal and added to this dataframe. 

In [7]:
df_with_props = df[df['material_density'].notna()]
df_with_props.head()

,Date,File Name,Run Time,Error Message,Velocity OK,Velocity at Max Compression,Time at Max Compression,Velocity at Max Compression Freq Uncertainty,Velocity at Max Compression Vel Uncertainty,Carrier Frequency,...,HEL Strain Rate,Peak Shock Stress,igsn,itemId,runId,alpss_dagster_version,alpss_version,material_c0,material_c_l,material_density
179,Jun 11 2026 05:04 PM,C1--20251022--00002.csv,0 days 00:00:00.928940,hel: rejected - HEL detected velocity < thresh...,True,248.387470,0.000002,2.141442e+06,1.659617,1.940000e+09,...,NaN,3.910202e+09,JHAMAB00021-16,6a2972a540fe08cc73d120e9,ef691e86-7c0f-4cce-9547-a67f11ad313b,0.1.1,1.7.1,3726,4430,8450
180,Jun 11 2026 05:04 PM,C1--20251022--00003.csv,0 days 00:00:00.926565,NaN,True,216.532972,0.000002,2.408179e+06,1.866339,1.864000e+09,...,0.000302,3.408738e+09,JHAMAB00021-16,6a2972ad40fe08cc73d1215f,ef691e86-7c0f-4cce-9547-a67f11ad313b,0.1.1,1.7.1,3726,4430,8450
181,Jun 11 2026 05:04 PM,C1--20251022--00005.csv,0 days 00:00:01.185289,NaN,True,217.658868,0.000002,3.064392e+06,2.374904,1.844058e+09,...,0.000345,3.426462e+09,JHAMAB00021-16,6a2972bc40fe08cc73d1222b,ef691e86-7c0f-4cce-9547-a67f11ad313b,0.1.1,1.7.1,3726,4430,8450
182,Jun 11 2026 05:04 PM,C1--20251022--00001.csv,0 days 00:00:00.993711,hel: rejected - HEL detected velocity < thresh...,True,246.973746,0.000002,2.184942e+06,1.693330,1.856058e+09,...,NaN,3.887947e+09,JHAMAB00021-16,6a29729d0b116ccbfdcc5ae7,ef691e86-7c0f-4cce-9547-a67f11ad313b,0.1.1,1.7.1,3726,4430,8450
183,Jun 11 2026 05:05 PM,C1--20251022--00008.csv,0 days 00:00:00.932085,hel: rejected - HEL detected velocity < thresh...,True,207.178434,0.000002,4.172703e+06,3.233845,1.840000e+09,...,NaN,3.261475e+09,JHAMAB00021-16,6a2972d30b116ccbfdcc5d05,ef691e86-7c0f-4cce-9547-a67f11ad313b,0.1.1,1.7.1,3726,4430,8450


### Group by Sample

See how many results per sample (IGSN):

In [8]:
df.groupby('igsn').size().sort_values(ascending=False).head(10)

igsn
JHAMAL00016-002       75
JHAMAA00004           62
JHAMAA00001-S5R4C3    60
JHAMAL00016-005       49
JHAMAL00016-013       45
JHAMAL00016-004       44
JHAMAL00018-006       44
JHAMAB00022-06        41
JHAMAL00018-007       40
JHAMAB00020-01        40
dtype: int64

### Summary Statistics

Velocity range for passing results:

In [9]:
df_passing['Velocity at Max Compression'].describe()

count    1100.000000
mean      309.505553
std       176.967581
min        25.019996
25%       181.049121
50%       303.960904
75%       404.242746
max      1630.516445
Name: Velocity at Max Compression, dtype: float64

### Handle Missing Values

Some columns may have NaN for certain experiments (e.g., HEL columns if HEL wasn't detected). Check coverage:

In [10]:
# Show columns with any NaN values
nan_counts = df.isna().sum()
nan_counts[nan_counts > 0].sort_values(ascending=False).head(10)

HEL Strain Rate               3125
HEL Uncertainty (GPa)         3064
HEL Strength (GPa)            3064
Strain Rate Uncertainty       2752
Spall Strength                2752
Velocity at Max Tension       2752
Strain Rate                   2752
Spall Strength Uncertainty    2752
Time at Recompression         2752
Velocity at Recompression     2752
dtype: int64